<a href="https://colab.research.google.com/github/aa-ahmed-arif/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aa-ahmed-arif/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use a **Decision Tree classifier** for this lane. The goal is to rank content pages by their likelihood of being declining, so that a content team can review higher-priority pages first.

A Decision Tree is appropriate because it is simple to inspect, can capture non-linear relationships, and is easier to interpret than a highly complex model. I will use a small maximum depth to reduce overfitting.

The target is is_declining_label. I will compare the model with my Week-4 hand-written baseline using **Precision@50**, because the practical decision is which pages should receive limited review capacity.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

I will use an **80/20 stratified train/test split** with a fixed random seed of 42. Stratification keeps the proportion of declining and non-declining pages similar in both sets.

The test set will be used only for the final comparison between the Decision Tree and the Week-4 baseline. I will not use label-derived fields such as trend_direction or trend_pct as model features because they are directly related to the target.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# ML-08: Train Decision Tree and compare with Week-4 baseline

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Make sure we are in the repository
REPO_DIR = "/content/flyrank-ml-internship"
os.chdir(REPO_DIR)

# Load starter dataset
DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

# ---------------------------------------------------------
# Define the target
# ---------------------------------------------------------
# The starter dataset does not contain is_declining_label.
# We derive it from trend_direction.
# IMPORTANT: trend_direction and trend_pct are NOT features.

df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# ---------------------------------------------------------
# Features
# ---------------------------------------------------------
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "search_volume",
    "avg_position",
    "ctr",
    "content_age_days"
]

target_col = "is_declining"

model_df = df[feature_cols + [target_col, "client_id"]].copy()

# Remove rows with missing model inputs
model_df = model_df.dropna(subset=feature_cols + [target_col])

X = model_df[feature_cols]
y = model_df[target_col]

print("Modeling rows:", len(model_df))
print("Declining rows:", int(y.sum()))
print("Declining rate:", round(y.mean(), 4))

# ---------------------------------------------------------
# 80/20 stratified split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Test rows:", len(X_test))

# ---------------------------------------------------------
# Train shallow Decision Tree
# ---------------------------------------------------------
model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

# Probability of declining
test_scores = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Precision@50
# ---------------------------------------------------------
def precision_at_50(y_true, scores):
    result = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top50 = result.sort_values(
        "score",
        ascending=False
    ).head(50)

    return top50["actual"].mean()

model_p50 = precision_at_50(y_test, test_scores)

# ---------------------------------------------------------
# Week-4 baseline on the SAME test rows
# ---------------------------------------------------------
baseline_test = model_df.loc[X_test.index].copy()

stale_threshold = df["days_since_last_update"].median()
volume_threshold = df["impressions_90d"].median()

baseline_test["baseline_score"] = (
    (baseline_test["days_since_last_update"] >= stale_threshold).astype(int)
    +
    (baseline_test["impressions_90d"] >= volume_threshold).astype(int)
)

baseline_p50 = precision_at_50(
    baseline_test[target_col],
    baseline_test["baseline_score"]
)

# ---------------------------------------------------------
# Model vs baseline
# ---------------------------------------------------------
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Decision Tree"
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ]
})

print("\nMODEL VS BASELINE")
display(comparison)

# ---------------------------------------------------------
# Feature importance
# ---------------------------------------------------------
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nFEATURE IMPORTANCE")
display(importance)

Modeling rows: 27532
Declining rows: 15525
Declining rate: 0.5639

Training rows: 22025
Test rows: 5507

MODEL VS BASELINE


,method,precision_at_50
0,Week-4 baseline,0.66
1,Decision Tree,0.78



FEATURE IMPORTANCE


,feature,importance
1,impressions_90d,0.416920
5,content_age_days,0.373305
3,avg_position,0.135047
4,ctr,0.074728
0,days_since_last_update,0.000000
2,search_volume,0.000000


## 4. Errors and interpretation

The model is a ranking aid rather than an automatic refresh decision. Some highly ranked pages may not actually be declining, while some declining pages may receive lower scores.

I will compare the Decision Tree and the Week-4 baseline using Precision@50 on the same held-out test set. If the model scores higher, this provides evidence that the additional observable signals were useful on this test split. If it scores lower, the Week-4 baseline remains a useful benchmark.

The Decision Tree is intentionally shallow (max_depth=3) so that interpretability is prioritized over unnecessary complexity.

In [7]:
# ML-08: Error analysis and interpretation

# Create test-set results
test_results = model_df.loc[X_test.index].copy()
test_results["model_score"] = test_scores
test_results["predicted"] = (test_results["model_score"] >= 0.5).astype(int)

# False positives: predicted declining but actually not declining
false_positives = test_results[
    (test_results[target_col] == 0) &
    (test_results["predicted"] == 1)
]

# False negatives: actually declining but not predicted declining
false_negatives = test_results[
    (test_results[target_col] == 1) &
    (test_results["predicted"] == 0)
]

print("Test rows:", len(test_results))
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nTop 10 model-ranked rows:")
display(
    test_results.sort_values(
        "model_score",
        ascending=False
    )[
        [
            "days_since_last_update",
            "impressions_90d",
            "search_volume",
            "avg_position",
            "ctr",
            "content_age_days",
            target_col,
            "model_score"
        ]
    ].head(10)
)

print("\nFeature importance:")
display(importance)

print("\nInterpretation:")
print(
    "The Decision Tree achieved higher Precision@50 than the Week-4 "
    "baseline on the same held-out test split (0.78 vs 0.66). "
    "This is directional evidence that the additional observable signals "
    "used by the model were useful for ranking likely declining pages "
    "in this test sample."
)

print(
    "\nThe strongest measured signals were impressions_90d, "
    "content_age_days, and avg_position. "
    "Feature importance is directional and should not be interpreted "
    "as causal evidence."
)

print(
    "\nLimitations: the target is derived from trend_direction, so "
    "trend_direction and trend_pct were excluded from the features. "
    "The model is a decision-support ranking aid, not a guarantee "
    "that a page should be refreshed."
)

Test rows: 5507
False positives: 1842
False negatives: 182

Top 10 model-ranked rows:


,days_since_last_update,impressions_90d,search_volume,avg_position,ctr,content_age_days,is_declining,model_score
10318,20,601,0.0,20.5,0.00,147,1,0.692832
15561,104,15641,0.0,35.6,0.08,229,1,0.692832
15174,28,73,10.0,11.8,0.00,148,1,0.692832
168,20,17992,10.0,6.4,0.11,144,0,0.692832
17026,20,52,20.0,64.4,0.00,91,0,0.692832
17063,104,34257,10.0,8.1,0.14,236,1,0.692832
26532,20,114389,0.0,39.7,0.13,141,1,0.692832
25739,104,294,0.0,7.2,0.00,256,1,0.692832
23148,104,11598,50.0,30.9,0.03,236,1,0.692832
20989,20,1138,10.0,36.0,0.09,104,0,0.692832



Feature importance:


,feature,importance
1,impressions_90d,0.416920
5,content_age_days,0.373305
3,avg_position,0.135047
4,ctr,0.074728
0,days_since_last_update,0.000000
2,search_volume,0.000000



Interpretation:
The Decision Tree achieved higher Precision@50 than the Week-4 baseline on the same held-out test split (0.78 vs 0.66). This is directional evidence that the additional observable signals used by the model were useful for ranking likely declining pages in this test sample.

The strongest measured signals were impressions_90d, content_age_days, and avg_position. Feature importance is directional and should not be interpreted as causal evidence.

Limitations: the target is derived from trend_direction, so trend_direction and trend_pct were excluded from the features. The model is a decision-support ranking aid, not a guarantee that a page should be refreshed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.